# 🎙️ Supertonic Voice Cloner
**Clone any voice in 3 steps — get a `.json` file to use in [Supertonic Voice Studio](https://github.com/CodekinsTech/supertonic-voice-studio)**

### How it works
1. **Upload** a 5-30 second clear voice recording (WAV/MP3)
2. **Train** — the optimizer matches a voice style to your recording (~10-20 min)
3. **Download** the `.json` voice file → upload it in Supertonic's Custom Voice panel

---
⚡ **GPU Required** — Go to `Runtime → Change runtime type → T4 GPU` before running

> **Disclaimer:** By using this tool you agree not to clone voices without the speaker's consent. You are solely responsible for how you use cloned voices.

## Step 1: Setup (run once)

In [ ]:
# Install dependencies and download Supertonic 3 model
!git clone https://github.com/saurabhv749/supertonic3-voice-clone.git /content/cloner 2>/dev/null || echo 'Already cloned'
%cd /content/cloner

!pip install --upgrade huggingface_hub -q
!huggingface-cli download Supertone/supertonic-3 --local-dir supertonic3 --quiet
!pip install -q -r requirements.txt

print('\n✅ Setup complete! Proceed to Step 2.')

## Step 2: Upload your voice recording
Upload a **5-30 second** clear voice clip (WAV or MP3). Avoid background noise/music.

In [ ]:
import os
from google.colab import files

os.makedirs('voices', exist_ok=True)

print('📁 Upload your voice recording (WAV or MP3)...')
uploaded = files.upload()

voice_file = list(uploaded.keys())[0]
voice_path = f'voices/{voice_file}'
os.rename(voice_file, voice_path)

# Convert to WAV if MP3
if voice_path.endswith('.mp3'):
    !pip install pydub -q
    from pydub import AudioSegment
    audio = AudioSegment.from_mp3(voice_path)
    voice_path = voice_path.replace('.mp3', '.wav')
    audio.export(voice_path, format='wav')
    print(f'Converted to WAV: {voice_path}')

# Preview
from IPython.display import Audio, display
print(f'\n🎧 Your uploaded voice:')
display(Audio(voice_path))
print(f'\n✅ Voice loaded: {voice_path}')

## Step 3: Clone the voice
This trains a voice style that matches your recording. Takes **10-20 minutes** on a T4 GPU.

- `VOICE_NAME`: name for your cloned voice (no spaces)
- `NUM_STEPS`: more steps = better quality but slower (1500-3000 recommended)

In [ ]:
#@title Training Settings
VOICE_NAME = 'my-voice' #@param {type:"string"}
NUM_STEPS = 2000 #@param {type:"slider", min:500, max:5000, step:500}
LEARNING_RATE = 0.0002 #@param {type:"number"}

!python train_style.py \
  --name "{VOICE_NAME}" \
  --target-wav-path "{voice_path}" \
  --num-steps {NUM_STEPS} \
  --learning-rate {LEARNING_RATE}

print(f'\n✅ Voice cloning complete!')

## Step 4: Preview your cloned voice

In [ ]:
import glob

# Find the best style file
style_files = sorted(glob.glob(f'logs/{VOICE_NAME}/*.json'))
best_style = style_files[-1] if style_files else None

if best_style:
    print(f'Using style: {best_style}')
    !python generate.py \
      --text "Hello! This is my cloned voice speaking. I hope it sounds natural and clear." \
      --style "{best_style}" \
      --speed 1.0

    # Play the generated audio
    samples = sorted(glob.glob('samples/*.wav'))
    if samples:
        print('\n🎧 Your cloned voice:')
        display(Audio(samples[-1]))
else:
    print('❌ No style file found. Run Step 3 first.')

## Step 5: Download the voice file
Download the `.json` voice style file, then upload it in **Supertonic Voice Studio → Custom Voice → Upload .json Voice**

In [ ]:
if best_style:
    print(f'📥 Downloading: {best_style}')
    files.download(best_style)
    print('\n✅ Done! Upload this .json file in Supertonic Voice Studio.')
else:
    print('❌ No style file found. Run Step 3 first.')